# 01. 데이터 수집 — Materials Project API
**목표:** 무기 화합물의 조성·구조 정보와 밴드갭(band gap) 데이터를 수집한다.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.getenv('MP_API_KEY')
print('API 키 로드:', '성공' if API_KEY else '실패')

API 키 로드: 성공


In [2]:
from mp_api.client import MPRester
import pandas as pd

# 산화물 계열 소재, 밴드갭 0~6 eV 범위로 수집
# 반도체(~0~3 eV) + 절연체(>3 eV) 포함 → 분류 해석에 풍부
with MPRester(API_KEY) as mpr:
    results = mpr.materials.summary.search(
        band_gap=(0.0, 6.0),
        fields=[
            'material_id',
            'formula_pretty',
            'band_gap',
            'elements',
            'nelements',
            'nsites',
            'volume',
            'density',
            'formation_energy_per_atom',
            'energy_above_hull',
            'is_stable',
        ],
        chunk_size=1000,
    )

print(f'수집된 소재 수: {len(results)}')

Retrieving SummaryDoc documents:   0%|          | 0/153328 [00:00<?, ?it/s]

수집된 소재 수: 153328


In [3]:
# DataFrame으로 변환
rows = []
for r in results:
    rows.append({
        'material_id': r.material_id,
        'formula': r.formula_pretty,
        'band_gap': r.band_gap,
        'elements': [str(e) for e in r.elements],
        'nelements': r.nelements,
        'nsites': r.nsites,
        'volume': r.volume,
        'density': r.density,
        'formation_energy_per_atom': r.formation_energy_per_atom,
        'energy_above_hull': r.energy_above_hull,
        'is_stable': r.is_stable,
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(153328, 11)


,material_id,formula,band_gap,elements,nelements,nsites,volume,density,formation_energy_per_atom,energy_above_hull,is_stable
0,mp-10018,Ac,0.0,[Ac],1,1,46.205987,8.157868,0.021639,0.021639,False
1,mp-862690,Ac,0.0,[Ac],1,4,184.545401,8.170182,0.000000,0.000000,True
2,mp-1183057,Ac,0.0,[Ac],1,3,136.026686,8.313274,0.015586,0.015586,False
3,mp-1183069,Ac,0.0,[Ac],1,3,138.841056,8.144760,0.011822,0.011822,False
4,mp-861724,Ac2AgIr,0.0,"[Ac, Ag, Ir]",3,4,110.157369,11.367264,-0.413285,0.087118,False


In [4]:
# 기본 통계
print(df['band_gap'].describe())
print('\n결측값:\n', df.isnull().sum())

count    153328.000000
mean          1.021404
std           1.437351
min           0.000000
25%           0.000000
50%           0.068200
75%           1.817800
max           5.999700
Name: band_gap, dtype: float64

결측값:
 material_id                  0
formula                      0
band_gap                     0
elements                     0
nelements                    0
nsites                       0
volume                       0
density                      0
formation_energy_per_atom    4
energy_above_hull            4
is_stable                    0
dtype: int64


In [5]:
# 데이터 저장 (API 재호출 없이 재사용)
df.to_csv('../data/raw_bandgap.csv', index=False)
print('저장 완료: data/raw_bandgap.csv')

저장 완료: data/raw_bandgap.csv
